## Imports

In [101]:
import os
import pandas as pd
import numpy as np
import cv2
import easyocr
from glob import glob
import tempfile
import shutil
from pathlib import Path


## Extraction of data

In [ ]:
def extractor(g2raw_path, helicoid_path, output_folder):
    output_file = os.path.join(output_folder, "angio_frames.png")
    cmd = f'{helicoid_path} "{g2raw_path}" --angio-unique-ident "{output_file}"'
    print("Running command")
    os.system(cmd)


In [85]:
def clahe(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    c = cv2.createCLAHE(2.0, (8, 8)).apply(g)
    return cv2.cvtColor(c, cv2.COLOR_GRAY2BGR)

In [86]:
def invert(img):
    return 255 - img

In [87]:
def upscale(img, scale=2):
    h, w = img.shape[:2]
    return cv2.resize(img, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)

In [ ]:
def detect(img, conf=0.15):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hits = []
    for b, t, c in reader.readtext(rgb):
        if c >= conf:
            hits.append(b)
    return hits


Using CPU. Note: This module is much faster with a GPU.


In [89]:
def mask_from_boxes(shape, boxes, pad=4):
    mask = np.zeros(shape[:2], dtype=np.uint8)
    for b in boxes:
        pts = np.array(b, dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)

    k = 2*pad + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.dilate(mask, kernel, 1)

In [ ]:
# def recall_first_mask(img):
#     masks = []

#     # OCR passes
#     for variant in [
#         img,
#         clahe(img),
#         invert(img),
#         upscale(img, 2)
#     ]:
#         boxes = detect(img)
#         m = mask_from_boxes(img.shape, boxes)
#         masks.append(m)

#     # Fixed overlay regions
#     #masks.append(fixed_roi_mask(img.shape))

#     final = np.maximum.reduce(masks)
#     return final

# def anonymize(image, blur_passes = 2):
#     mask = recall_first_mask(image)
#     for i in range(blur_passes):
#         blurred = cv2.GaussianBlur(image, (0,0), 7)
#     out = image.copy()
#     out[mask == 255] = blurred[mask == 255]
#     return out, mask

# Same output for without preprocessing the anonymizing happens in 7-8 seconds per image, but with preprocessing it takes 30 seconds.

In [94]:
def anonymize(image, blur_passes = 2):
    boxes = detect(image)
    mask = mask_from_boxes(image.shape, boxes)
    for i in range(blur_passes):
        blurred = cv2.GaussianBlur(image, (0,0), 7)
    out = image.copy()
    out[mask == 255] = blurred[mask == 255]
    return out, mask

In [95]:
image = cv2.imread('test.png')
a, b = anonymize(image)


c:\Users\dbhagria\AppData\Local\miniconda3\envs\envrionment\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [96]:
cv2.imwrite("test3.png", a)

True

In [ ]:
def batch_anonymize(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    # get all image files (png, jpg, etc.)
    image_files = sorted(glob(os.path.join(input_folder, "*.*")))

    for path in image_files:
        img = cv2.imread(path)
        if img is None:
            print(f"Skipping {path}, could not read image.")
            continue

        # anonymize
        anon_img, mask = anonymize(img)

        # save anonymized image
        base_name = os.path.basename(path)
        save_path = os.path.join(output_folder, base_name)
        cv2.imwrite(save_path, anon_img)

        print(f"Anonymized: {base_name}")

    print("All images processed and saved to:", output_folder)


In [ ]:
def secure_processing_pipeline(g2raw_path, helicoid_path, output_folder):
    """
    Steps:
    1. Create temp folder (PHI stored here)
    2. Extract raw images into temp
    3. Anonymize → verification folder
    4. Wait for user to approve
    5. Delete PHI temp data automatically
    """

    with tempfile.TemporaryDirectory() as temp_phi_dir:
        print("TEMP PHI DIR:", temp_phi_dir)

        # Step 1: extract frames with PHI
        extractor(g2raw_path, helicoid_path, temp_phi_dir)

        # Step 2: anonymize into safe folder
        batch_anonymize(temp_phi_dir, output_folder)

        print("\nReview anonymized images here:")
        print(os.path.abspath(output_folder))

        input("\nPress ENTER to delete original PHI images...")

        # leaving `with` automatically deletes temp dir
        print("PHI temporary directory securely deleted.")

In [ ]:
def main():
    VERIFICATION_FOLDER = 'review_anonymized'
    reader = easyocr.Reader(['en'], gpu=False)
    os.makedirs(VERIFICATION_FOLDER, exist_ok=True)

    g2raw_path = r"S:\01 - Clinical G2 Data\2022 - CFD\04-058 - CFD\04-058 - OCT\04-058-Anonymous-Anonymous-20240329-093034.g2raw"
    helicoid_path = r"C:\Users\dbhagria\Genshi-v25.5.5\bin\helicoid-magick.exe"
    secure_processing_pipeline(g2raw_path, helicoid_path, VERIFICATION_FOLDER)

Running command


In [102]:
if __name__ == "__main__":
    main()

Running command
